# Supervised Fine-Tuning (SFT) with Serverless customization on SageMaker AI

## Lab 3 — Benchmark Evaluation

In this notebook, we evaluate the SFT fine-tuned model using a **benchmark evaluation** and compare it against the base model.

### What you'll do

1. Retrieve the fine-tuned model from the Model Registry
2. Explore available benchmarks and create a `BenchMarkEvaluator`
3. Run evaluation on both the fine-tuned **and** base model
4. Compare the results

---

## 1. Prerequisites

### Set up the SageMaker session

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

### Retrieve the fine-tuned model

In [ ]:
from sagemaker.core.resources import ModelPackageGroup
from config import BASE_MODEL_ID

model_package_group_name = f"{BASE_MODEL_ID}-sft-mpg"
model_package_version = "1"

model_package_group = ModelPackageGroup.get(model_package_group_name)

fine_tuned_model_package_arn = f"{model_package_group.model_package_group_arn.replace('model-package-group', 'model-package', 1)}/{model_package_version}"
print(f"Fine-tuned Model Package ARN: {fine_tuned_model_package_arn}")

In [ ]:
# Get BaseModel info from the fine-tuned model package
ft_response = sm_client.describe_model_package(
    ModelPackageName=fine_tuned_model_package_arn
)
base_model_info = ft_response['InferenceSpecification']['Containers'][0]['BaseModel']
base_model_id = base_model_info['HubContentName']

print(f"Base model ID: {base_model_id}")
print(base_model_info)

if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{base_model_id}/benchmark-evaluation"
else:
    output_path = f"s3://{bucket_name}/{base_model_id}/benchmark-evaluation"

if default_prefix:
    output_path_base = f"s3://{bucket_name}/{default_prefix}/{base_model_id}/benchmark-evaluation-base"
else:
    output_path_base = f"s3://{bucket_name}/{base_model_id}/benchmark-evaluation-base"

print(f"Evaluation base output path: {output_path_base}")
print(f"Evaluation output path: {output_path}")

---

## 2. Explore available benchmarks

In [ ]:
from sagemaker.train.evaluate import BenchMarkEvaluator, get_benchmarks, get_benchmark_properties
from rich.pretty import pprint

Benchmark = get_benchmarks()
pprint(list(Benchmark))

Inspect the benchmark we will use. Change `Benchmark.MEDICAL` to whichever benchmark is most relevant from the list above (e.g. `Benchmark.MMLU`, `Benchmark.MATH`, etc.).

In [ ]:
# Pick the benchmark that best fits your use case from the list above.
# For a medical SFT model, look for a medical benchmark (e.g. MEDICAL, MMLU).
# If no medical-specific benchmark exists, GPQA is a good general choice.
SELECTED_BENCHMARK = Benchmark.GPQA

pprint(get_benchmark_properties(benchmark=SELECTED_BENCHMARK))

---

## 3. Run benchmark evaluation

We run two separate benchmark evaluations — one for the fine-tuned model and one for the base model — then compare the results.

In [ ]:
# Evaluate the fine-tuned model
evaluator = BenchMarkEvaluator(
    benchmark=SELECTED_BENCHMARK,
    model=fine_tuned_model_package_arn,
    model_package_group=model_package_group_name,
    base_eval_name='fine-tuned-model-sft',
    s3_output_path=output_path,
    evaluate_base_model=False,
)

execution = evaluator.evaluate()
#execution.wait()

# Evaluate the base model using the JumpStart model ID directly
base_evaluator = BenchMarkEvaluator(
    benchmark=SELECTED_BENCHMARK,
    model=base_model_id,
    base_eval_name='base-model-sft',
    s3_output_path=output_path_base,
)

execution_base = base_evaluator.evaluate()
execution_base.wait()

## 4. Evaluation Results

In [ ]:
from sagemaker.train.evaluate import EvaluationPipelineExecution
from sagemaker.train.evaluate.constants import EvalType

# Get all succeeded evaluations and take the first 2
all_succeeded = [
    e for e in EvaluationPipelineExecution.get_all(eval_type=EvalType.BENCHMARK)
    if e.status.overall_status == "Succeeded"
]

last_two_succeeded = all_succeeded[:2]

# Print both
for i, execution in enumerate(last_two_succeeded, 1):
    print(f"=== Succeeded Evaluation #{i} ===")
    pprint(execution)
    pprint(execution.show_results())